In [1]:
import os
import re
import time
import json
import torch
import commons
import utils
from models import SynthesizerTrn
from text_JP import cleaned_text_to_sequence, symbols
import pyopenjtalk
from text_JP.phonemize import Phonemizer

# This script is designed to be run in a Jupyter Notebook or an environment
# with IPython display capabilities.
from IPython.display import Audio, display

/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


# 1 全体デコード 

# 2 細切れデコード　overlapなし 

# 3 細切れデコード　overlapあり

# 4 細切れデコード　overlapあり　相互相関で調整

# 5 細切れデコード　スペクトログラムをoverlapで接合　相互相関で調整はしない

# 6 細切れデコード　スペクトログラムを接合　overlapあり　相互相関で調整あり

# 共通クラス　設定

In [2]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import commons
import utils
from models import SynthesizerTrn
from text_JP import symbols
from scipy.io.wavfile import write

# ==========================================
# 1. 設定
# ==========================================
config_path = "./logs/uudb_csj21/config.json"
checkpoint_path = "./logs/uudb_csj21/G_3020000.pth"
input_txt_path = "./filelists/csj_uudb_test_fine.txt"
output_dir = "output_wavs_batch"

# 生成パラメータ
noise_scale = 1.0
noise_scale_w = 1.0
length_scale = 1.0

# デバイス設定
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


# Load configuration
hps = utils.get_hparams_from_file(config_path)

# Load model
print("Loading model...")
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model).to(device)
    
    # Set model to evaluation mode
_ = net_g.eval()
    
    # Load checkpoint
print(f"Loading checkpoint from {checkpoint_path}...")
_ = utils.load_checkpoint(checkpoint_path, net_g, None)

# ==========================================
# 2. 共通クラス・関数定義
# ==========================================

class TorchSTFT(torch.nn.Module):
    def __init__(self, filter_length=800, hop_length=200, win_length=800, window='hann'):
        super().__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        self.win_length = win_length
        # windowのデバイス転送はinverse内で行うためここでは作成のみ
        self.window = torch.hann_window(win_length, periodic=True)

    def inverse(self, magnitude, phase):
        complex_spec = magnitude * torch.exp(phase * 1j)
        # istftは (..., Freq, Time) を期待するため、Batch次元等がある場合は維持される
        inverse_transform = torch.istft(
            complex_spec,
            self.filter_length, self.hop_length, self.win_length, 
            window=self.window.to(complex_spec.device)
        )
        return inverse_transform.unsqueeze(1)

def find_best_frame_shift(ref_spec, target_spec, search_range=5):
    # 4次元(B, S, F, T)対応: 全サブバンド・周波数をまとめて相関をとる
    if ref_spec.dim() == 4:
        b, s, f, t = ref_spec.shape
        ref_spec = ref_spec.reshape(b, s * f, t).contiguous()
        target_spec = target_spec.reshape(b, s * f, t).contiguous()
    
    ref_log = torch.log(ref_spec + 1e-6)
    target_log = torch.log(target_spec + 1e-6)
    ref_log = ref_log - torch.mean(ref_log, dim=-1, keepdim=True)
    target_log = target_log - torch.mean(target_log, dim=-1, keepdim=True)

    pad_target = F.pad(target_log, (search_range, search_range))
    cross_corr = F.conv1d(pad_target, ref_log)
    max_idx = torch.argmax(cross_corr)
    return max_idx.item() - search_range

def get_text_from_phonemes(phonemes, hps):
    symbol_to_id = {s: i for i, s in enumerate(symbols)}
    clean_phonemes = phonemes.replace("[", "").replace("]", "").strip()
    phoneme_list = clean_phonemes.split(" ")
    text_norm = []
    for p in phoneme_list:
        if p in symbol_to_id:
            text_norm.append(symbol_to_id[p])
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    return torch.LongTensor(text_norm)

def istft_finalize(net_g, full_complex_spec):
    """
    Multi-stream iSTFTに対応した波形再構成関数
    full_complex_spec shape: [Batch, Subbands, Freq, Time]
    """
    device = full_complex_spec.device
    final_spec = torch.abs(full_complex_spec)
    final_phase = torch.angle(full_complex_spec)
    
    stft = TorchSTFT(
        filter_length=net_g.dec.gen_istft_n_fft, 
        hop_length=net_g.dec.gen_istft_hop_size, 
        win_length=net_g.dec.gen_istft_n_fft
    ).to(device)

    # Multi-stream処理の判定
    if hasattr(net_g.dec, 'subbands') and net_g.dec.subbands > 1:
        # [B, S, F, T] -> [B*S, F, T] に変形してiSTFT
        b, s, f, t = final_spec.shape
        spec_reshaped = final_spec.view(b * s, f, t)
        phase_reshaped = final_phase.view(b * s, f, t)
        
        y_mb_hat = stft.inverse(spec_reshaped, phase_reshaped) # -> [B*S, 1, Time_sub]
        y_mb_hat = y_mb_hat.squeeze(1).view(b, s, -1)          # -> [B, S, Time_sub]

        # 合成フィルタ (Synthesis Filter Bank)
        if net_g.ms_istft_vits:
            # 学習済みアップサンプリングフィルタを使用
            y_mb_hat = F.conv_transpose1d(
                y_mb_hat, 
                net_g.dec.updown_filter.to(device) * net_g.dec.subbands, 
                stride=net_g.dec.subbands
            )
            audio_tensor = net_g.dec.multistream_conv_post(y_mb_hat)
        else:
            # PQMFまたは単純加算 (Fallback)
            try:
                from pqmf import PQMF
                pqmf = PQMF(device)
                audio_tensor = pqmf.synthesis(y_mb_hat.unsqueeze(2)) 
            except ImportError:
                 audio_tensor = torch.sum(y_mb_hat, dim=1, keepdim=True)
    else:
        # 通常のiSTFT (Single stream)
        audio_tensor = stft.inverse(final_spec, final_phase)

    return audio_tensor[0, 0].data.cpu().float().numpy()

Using device: cuda
Loading model...
Mutli-stream iSTFT VITS


/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading checkpoint from ./logs/uudb_csj21/G_3020000.pth...


/home/synology/naru/work/ms2/MB-iSTFT-VITS/utils.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')


## new cond1

モーラ単位で分割g2p

In [7]:
def synthesize_cond3_auto_mora(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2, mora_chunk_size=1):
    """
    Cond 3: 相互相関スペクトログラムOLA (全コンテキストエンコード + モーラ分割デコード)
    
    1. テキスト全体 -> 全体z生成 (Full Context)
    2. zをモーラ単位に分割（ただし隣接区間でオーバーラップを持たせる）
    3. 個別にデコード -> 相互相関で位置合わせしてクロスフェード -> iSTFT
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # --- A. テキスト解析 ---
    chunk_phonemes_list = get_mora_chunks_from_text(raw_text, phonemizer)
    
    mora_phoneme_counts = []
    all_phoneme_ids = []
    for ph_str in chunk_phonemes_list:
        ids = get_text_from_phonemes(ph_str, hps)
        mora_phoneme_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids:
        return np.array([])

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # --- B. 全体エンコード ---
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        # --- C. チャンク単位で分割デコード & OLA ---
        full_complex_spec = None
        prev_raw_tail = None
        
        current_ph_idx = 0
        current_z_frame = 0
        ratio = None 
        
        num_chunks = len(mora_phoneme_counts)
        
        # mora_chunk_size ずつまとめてループ
        for i in range(0, num_chunks, mora_chunk_size):
            chunk_counts = mora_phoneme_counts[i : i + mora_chunk_size]
            total_phonemes_in_chunk = sum(chunk_counts)
            
            # 1. このチャンク(複数モーラ)の正味の長さ
            chunk_durations = w_ceil_flat[current_ph_idx : current_ph_idx + total_phonemes_in_chunk]
            chunk_z_len = int(torch.sum(chunk_durations).item())
            
            # 2. 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + chunk_z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            
            if z_end_decode > z.shape[2]:
                z_end_decode = z.shape[2]
            
            # z 切り出し
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            # デコード
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # --- OLA処理 (前回と同じロジック) ---
                actual_z_overlap = max(0, z_chunk.shape[-1] - chunk_z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None: 
                        prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else:
                        prev_ref = prev_raw_tail
                    
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(
                                torch.abs(prev_ref[..., :valid_overlap]), 
                                torch.abs(curr_ref[..., :valid_overlap]), 
                                search_range
                            )
                        
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            
                            full_complex_spec = torch.cat([
                                full_complex_spec[..., :-cross_len], 
                                merged, 
                                aligned[..., cross_len:]
                            ], dim=-1)
                        else:
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                # 末尾保存
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            # ポインタ更新
            current_ph_idx += total_phonemes_in_chunk
            current_z_frame = z_end_nominal # 正味分だけ進める
            
            if current_z_frame >= z.shape[2]:
                break

        # --- D. iSTFT ---
        if full_complex_spec is None:
            return np.array([])
            
        audio = istft_finalize(net_g, full_complex_spec)
        return audio

In [8]:
import time

# ==========================================
# 0. 計測用ヘルパー関数
# ==========================================
def measure_and_print(label, func, *args):
    """
    関数を実行し、所要時間・音声長・RTFを計測して表示する
    """
    # GPUの処理待ちをリセット
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    # 関数の実行
    audio = func(*args)
    
    # GPU処理完了まで待機
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    elapsed_time = time.time() - start_time
    
    # 音声の長さを計算 (秒)
    if len(audio) > 0:
        duration = len(audio) / hps.data.sampling_rate
        rtf = elapsed_time / duration
    else:
        duration = 0
        rtf = 0

    print(f"--- {label} ---")
    print(f"Audio duration: {duration:.2f} seconds")
    print(f"Elapsed time: {elapsed_time:.4f} seconds")
    print(f"Real Time Factor (RTF): {rtf:.4f}")
    print("-" * 30)
    
    return audio

def japanese_cleaner_revised(text):
    parts = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    phoneme_parts = []
    phonemizer = Phonemizer()
    for part in parts:
        if not part or part.isspace():
            continue
        if part.startswith('[') and part.endswith(']') and len(part) > 2:
            content = part[1:-1]
            if not content:
                phoneme_parts.append('[ ]')
            else:
                kana_content = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                phoneme_content = phonemizer(kana_content)
                phoneme_parts.append(f'[ {phoneme_content} ]')
            continue
        if part == '{cough}' or part == '<cough>':
            phoneme_parts.append('<cough>')
            continue
        if part in '、。':
            phoneme_parts.append('sp')
            continue
        kana = pyopenjtalk.g2p(part, kana=True).replace('ヲ', 'オ')
        phonemes = phonemizer(kana)
        phoneme_parts.append(phonemes)
    final_text = ' '.join(phoneme_parts)
    return re.sub(r'\s+', ' ', final_text).strip()

def text_to_sequence_custom(text, hps):
    phonemized_text = japanese_cleaner_revised(text)
    stn_tst = cleaned_text_to_sequence(phonemized_text)
    if hps.data.add_blank:
        stn_tst = commons.intersperse(stn_tst, 0)
    return torch.LongTensor(stn_tst)

# ==========================================
# 比較用: Cond 4 (全体一括生成 / Baseline)
# ==========================================
def synthesize_cond4_full(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 4: 全体一括生成 (Baseline)
    分割せず、テキスト全体をそのまま生成する
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    
    # G2P (全体)
    # 既存のロジックを使って音素化 (特殊記号などは考慮しつつ、分割はしない)
    # ここでは簡易的に結合したものを使用、または phonemizer で全体を一括処理
    # 文脈を維持するため、全体を1つのシーケンスとする
    
    # 簡易実装: 一度 chunk に分けたものを結合して再構成
    # (厳密には original_transcriptions_summary のテキストを直接 g2p するのがベスト)
    
    # ここでは既存の text_to_phoneme 相当のことを行う
    #import pyopenjtalk
    #kana = pyopenjtalk.g2p(raw_text, kana=True).replace('ヲ', 'オ')
    #phonemes = phonemizer(kana)
    stn_tst = text_to_sequence_custom(raw_text, hps)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        audio = net_g.infer(
            x_tst, x_tst_lengths, sid=sid, 
            noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
        )[0][0,0].data.cpu().float().numpy()
        
    return audio


# 文節単位で

In [3]:
import re
import pyopenjtalk

def get_bunsetsu_chunks_strict(text, phonemizer):
    """
    厳密な文節分割:
    - タグ [...] と通常テキストの境界で分割
    - 句読点 (、。) は直前の要素に結合して分割確定
    例: "でも[なんか]、そうすると" -> ["でも", "[なんか]、", "そうすると"]
    """
    # 1. トークン化: タグ、句読点、それ以外のテキストに分解
    # {cough} 等もタグ扱い
    tokens = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    
    chunks_phonemes = []
    current_chunk_text = ""
    current_type = None # 'text' or 'tag'
    
    def finalize_chunk(txt):
        if not txt: return None
        
        # 音素化処理
        # 句読点が含まれているかチェックして sp に変換する処理が必要
        
        # 簡易パース: 末尾が句読点かどうか
        has_punct = False
        if txt.endswith("、") or txt.endswith("。"):
            has_punct = True
            content = txt[:-1] # 句読点除去
        else:
            content = txt
            
        if not content and not has_punct: return None
        
        # 中身の音素化
        ph_str = ""
        if content:
            # タグの場合
            if content.startswith("[") and content.endswith("]"):
                inner = content[1:-1]
                if inner:
                    k = pyopenjtalk.g2p(inner, kana=True).replace('ヲ', 'オ')
                    p = phonemizer(k)
                    ph_str = f"[ {p} ]"
                else:
                    ph_str = "[ ]"
            elif content in ["{cough}", "<cough>"]:
                ph_str = "<cough>"
            else:
                # 通常テキスト
                k = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                p = phonemizer(k)
                ph_str = p
        
        # 句読点があれば sp を付与
        if has_punct:
            if ph_str:
                ph_str += " sp"
            else:
                ph_str = "sp"
                
        return ph_str.strip()

    for token in tokens:
        if not token or token.isspace():
            continue
            
        # トークンの種類判定
        is_punct = token in ["、", "。"]
        is_tag = (token.startswith("[") and token.endswith("]")) or token in ["{cough}", "<cough>"]
        
        if is_punct:
            # 句読点 -> 現在のチャンクに結合して、即座に確定(Flush)
            current_chunk_text += token
            
            ph = finalize_chunk(current_chunk_text)
            if ph: chunks_phonemes.append(ph)
            
            current_chunk_text = ""
            current_type = None
            
        else:
            # タグまたはテキスト
            new_type = 'tag' if is_tag else 'text'
            
            # タイプが変わったら、前のチャンクを確定して切り離す
            # (例: "でも" (text) -> "[なんか]" (tag) の境界)
            if current_type is not None and current_type != new_type:
                ph = finalize_chunk(current_chunk_text)
                if ph: chunks_phonemes.append(ph)
                current_chunk_text = ""
            
            current_chunk_text += token
            current_type = new_type
            
    # 残りがあれば処理
    if current_chunk_text:
        ph = finalize_chunk(current_chunk_text)
        if ph: chunks_phonemes.append(ph)
        
    return chunks_phonemes

# mecab利用

In [4]:
import re
import MeCab
import pyopenjtalk

# ==========================================
# 1. MeCabによる文節分割ロジック
# ==========================================
def split_text_to_bunsetsu(text):
    """
    MeCabを用いてテキストを「文節」単位のリストに分割する。
    文節 = [自立語 + 付属語...] の塊
    """
    # unidic-lite を使用する場合の標準的なタガー初期化
    tagger = MeCab.Tagger()
    
    node = tagger.parseToNode(text)
    chunks = []
    current_chunk = ""
    
    while node:
        if node.surface == "":  # BOS/EOSスキップ
            node = node.next
            continue
            
        # 品詞情報を取得
        # UniDic系: pos1, pos2, pos3, pos4, ...
        # IPADIC系: pos1, pos2, ...
        features = node.feature.split(",")
        pos1 = features[0]  # 品詞大分類
        pos2 = features[1]  # 品詞中分類
        
        # --- 文節の切れ目判定 ---
        # 「自立語」が来たタイミングで分割（ただし文頭は除く）
        # ※定義は簡易的なものですが、音声合成の韻律単位としては概ね機能します
        is_independent = False
        
        # 分割候補となる品詞 (名詞, 動詞, 形容詞, 副詞, 連体詞, 接続詞, 感動詞, 接頭詞, 形状詞)
        if pos1 in ["名詞", "動詞", "形容詞", "副詞", "連体詞", "接続詞", "感動詞", "接頭詞", "形状詞", "代名詞"]:
            # ただし「非自立」「接尾」などは前の文節に付ける（分割しない）
            if pos2 not in ["非自立", "接尾"]:
                is_independent = True
                
        # 代名詞などの扱い（UniDicでは代名詞も独立しやすいが、IPADICでは名詞扱い）
        
        # 分割実行
        if is_independent and current_chunk:
            chunks.append(current_chunk)
            current_chunk = ""
            
        current_chunk += node.surface
        node = node.next
        
    # 残りを追加
    if current_chunk:
        chunks.append(current_chunk)
        
    return chunks

# ==========================================
# 2. タグや句読点処理との統合
# ==========================================
def get_bunsetsu_chunks_mecab(text, phonemizer):
    """
    タグ [...] や句読点を考慮しつつ、
    通常テキスト部分は MeCab で文節分割して音素化リストを返す
    """
    # 1. タグや句読点で大まかに分割
    tokens = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    
    final_phoneme_chunks = []
    
    # 一時バッファ（MeCabにかける前のテキスト）
    text_buffer = ""
    
    def flush_text_buffer():
        nonlocal text_buffer
        if not text_buffer: return
        
        # MeCabで文節分割
        bunsetsu_list = split_text_to_bunsetsu(text_buffer)
        
        for b_text in bunsetsu_list:
            # 音素化
            k = pyopenjtalk.g2p(b_text, kana=True).replace('ヲ', 'オ')
            p = phonemizer(k)
            if p.strip():
                final_phoneme_chunks.append(p.strip())
        
        text_buffer = ""

    for token in tokens:
        if not token or token.isspace():
            continue
            
        # A. 句読点 (直前の文節にくっつける -> sp化)
        if token in ["、", "。"]:
            # バッファがあれば先に吐き出す
            if text_buffer:
                # 最後の文節を取得して sp を付けるために、ここで自前処理
                bunsetsu_list = split_text_to_bunsetsu(text_buffer)
                for i, b_text in enumerate(bunsetsu_list):
                    k = pyopenjtalk.g2p(b_text, kana=True).replace('ヲ', 'オ')
                    p = phonemizer(k)
                    if p.strip():
                        # 最後の文節なら sp を付与
                        if i == len(bunsetsu_list) - 1:
                            final_phoneme_chunks.append(p.strip() + " sp")
                        else:
                            final_phoneme_chunks.append(p.strip())
                text_buffer = ""
            else:
                # 直前の既存チャンクに sp を追加
                if final_phoneme_chunks:
                    final_phoneme_chunks[-1] += " sp"
                else:
                    final_phoneme_chunks.append("sp")
            continue
            
        # B. タグ
        if (token.startswith("[") and token.endswith("]")) or token in ["{cough}", "<cough>"]:
            flush_text_buffer() # テキストがあれば先に処理
            
            # タグの処理
            if token.startswith("["):
                content = token[1:-1]
                if content:
                    k = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                    p = phonemizer(k)
                    final_phoneme_chunks.append(f"[ {p} ]")
                else:
                    final_phoneme_chunks.append("[ ]")
            else:
                final_phoneme_chunks.append("<cough>")
            continue
            
        # C. 通常テキスト -> バッファに溜める
        text_buffer += token
        
    # 残りのテキストを処理
    flush_text_buffer()
    
    return final_phoneme_chunks

In [5]:
def get_z_and_phoneme_durations(net_g, x_tst, x_tst_lengths, sid, noise_scale, noise_scale_w, length_scale):
    """
    修正版 (行列積の順序修正): テキスト全体から一括で潜在表現 z と、各音素の継続長を取得する
    """
    # 1. Text Encoder
    # x: [B, C, T_phoneme]
    # m_p: [B, C, T_phoneme], logs_p: [B, C, T_phoneme]
    x, m_p, logs_p, x_mask = net_g.enc_p(x_tst, x_tst_lengths)
    
    # 2. Speaker Embedding
    if net_g.n_speakers > 0:
        g = net_g.emb_g(sid).unsqueeze(-1) # [b, h, 1]
    else:
        g = None

    # 3. Duration Predictor (w_ceil を取得)
    logw = net_g.dp(x, x_mask, g=g)
    w = torch.exp(logw) * x_mask * length_scale
    w_ceil = torch.ceil(w) # [B, 1, T_phoneme]
    
    # 4. Expand (手動アライメントマスク作成)
    B, _, T_phoneme = w_ceil.shape
    T_frame = int(torch.sum(w_ceil).item())
    
    # attn_mask: [B, T_phoneme, T_frame]
    # 「音素iは、フレームjからkまでを担当する」というマスクを作る
    attn_mask = torch.zeros(B, T_phoneme, T_frame).to(x.device)
    
    # バッチサイズ1前提 (推論用)
    w_ceil_flat = w_ceil.squeeze() # [T_phoneme]
    current_frame = 0
    
    for i, dur in enumerate(w_ceil_flat):
        d = int(dur.item())
        if d > 0:
            # 音素 i に対応するフレーム範囲を 1.0 にする
            attn_mask[0, i, current_frame : current_frame + d] = 1.0
            current_frame += d
    
    # Expand mean & variance
    # m_p: [B, Channels, T_phoneme]
    # attn_mask: [B, T_phoneme, T_frame]
    # matmul -> [B, Channels, T_frame]
    m_p = torch.matmul(m_p, attn_mask)
    logs_p = torch.matmul(logs_p, attn_mask)

    # 5. Flow (Reverse) -> z を生成
    # y_mask (フレーム側のマスク) を作成: 全フレーム有効なので全て1
    y_mask = torch.ones(B, 1, T_frame).to(x.device)
    
    z_p = m_p + torch.randn_like(m_p, dtype=torch.float) * torch.exp(logs_p) * noise_scale
    z = net_g.flow(z_p, y_mask, g=g, reverse=True)
    
    return z, w_ceil, g

In [6]:
def synthesize_cond1_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 1: 文節単位単純接続 (Strict分割)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # ★変更: Strict分割を使用
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    audio_segments = []
    
    # 以下同様...
    for ph in bunsetsu_chunks:
        if not ph: continue
        stn_tst = get_text_from_phonemes(ph, hps)
        
        with torch.no_grad():
            x_tst = stn_tst.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
            
            audio = net_g.infer(
                x_tst, x_tst_lengths, sid=sid, 
                noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
            )[0][0,0].data.cpu().float().numpy()
            
        audio_segments.append(audio)
    
    if not audio_segments: return np.array([])
    return np.concatenate(audio_segments)

In [7]:
def synthesize_cond2_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 2: スペクトログラム単純接続 (Strict分割)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # ★変更: Strict分割
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    # 以下同様にリスト構築...
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([])
    
    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze() 

        full_complex_spec = None
        current_ph_idx = 0
        current_z_frame = 0
        
        for count in chunk_counts:
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([])
        return istft_finalize(net_g, full_complex_spec)

In [8]:
def synthesize_cond3_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2):
    """
    Cond 3: 相互相関OLA (全体エンコード -> 文節単位デコード + Overlap -> OLA)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # 1. 文節リスト & ID化
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([])

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # 2. 全体エンコード
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        # 3. 文節ごとに OLA 処理
        full_complex_spec = None
        prev_raw_tail = None
        current_ph_idx = 0
        current_z_frame = 0
        ratio = None 
        
        for count in chunk_counts:
            # 長さ計算
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # OLA (前回と同様のロジック)
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None:
                        prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else:
                        prev_ref = prev_raw_tail
                        
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(
                                torch.abs(prev_ref[..., :valid_overlap]), 
                                torch.abs(curr_ref[..., :valid_overlap]), 
                                search_range
                            )
                        
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            full_complex_spec = torch.cat([
                                full_complex_spec[..., :-cross_len], merged, aligned[..., cross_len:]
                            ], dim=-1)
                        else:
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                # 末尾保存
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            current_ph_idx += count
            current_z_frame = z_end_nominal # 正味分だけ進める
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([])
        return istft_finalize(net_g, full_complex_spec)

In [13]:

# ==========================================
# 実行準備
# ==========================================
# Phonemizerのインスタンス化 (ループの外で1回だけ行う)
phonemizer = Phonemizer()

# (以下、メインループ内での呼び出し例)
# ...
raw_japanese_text = "[あ]ちゃんと入ってないんだ 、 [あー]" #(original_transcriptions_summary.txt等から取得)
#x_tst = stn_tst.to(device).unsqueeze(0)
#x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
sid = torch.LongTensor([374]).to(device)
# ...
audio_cond1 = synthesize_cond1_bunsetsu_M(net_g, raw_japanese_text, sid, hps, phonemizer)
audio_cond2 = synthesize_cond2_bunsetsu_M(net_g, raw_japanese_text, sid, hps, phonemizer)
audio_cond3 = synthesize_cond3_bunsetsu_M(net_g, raw_japanese_text, sid, hps, phonemizer,1,2)
# write(..., audio_cond1)
print("条件１")
display(Audio(audio_cond1, rate=hps.data.sampling_rate, normalize=False))
print("条件２")
display(Audio(audio_cond2, rate=hps.data.sampling_rate, normalize=False))
print("条件３")
display(Audio(audio_cond3, rate=hps.data.sampling_rate, normalize=False))

条件１


条件２


条件３


In [9]:
import torch
import numpy as np

# ==========================================
# 1. 共通準備: 潜在表現と分割情報の生成
# ==========================================
def prepare_shared_latents(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 2, 3, 4 で共有する潜在表現 z と、文節分割情報を一括生成する
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # A. 文節分割 (MeCab版を使用)
    # ※ get_bunsetsu_chunks_mecab が定義済みであることを前提とします
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    # B. 全体の音素列IDと、各文節の音素数を取得
    all_phoneme_ids = []
    chunk_phoneme_counts = []
    
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_phoneme_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids:
        return None, None, None, None

    # C. 一括エンコード (Full Context)
    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # エンコーダ実行 (z, w_ceil, g を取得)
        # ※ get_z_and_phoneme_durations は修正済みのものを使用
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
    
    return z, w_ceil, g, chunk_phoneme_counts

# ==========================================
# 2. Cond 2: スペクトログラム単純接続 (共有z版)
# ==========================================
def synthesize_cond2_shared(net_g, z, w_ceil, g, chunk_counts):
    """
    共有された z を文節ごとに切り出し、デコードして単純結合する
    """
    if z is None: return np.array([])
    
    w_ceil_flat = w_ceil.squeeze() 
    full_complex_spec = None
    
    current_ph_idx = 0
    current_z_frame = 0
    
    with torch.no_grad():
        for count in chunk_counts:
            # この文節に対応するフレーム数を計算
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # z 切り出し
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            # デコード
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                # 単純結合
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            # ポインタ更新
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

    if full_complex_spec is None: return np.array([])
    return istft_finalize(net_g, full_complex_spec)

# ==========================================
# 3. Cond 3: 相互相関OLA (共有z版)
# ==========================================
def synthesize_cond3_shared(net_g, z, w_ceil, g, chunk_counts, z_overlap_frames=5, search_range=2):
    """
    共有された z をOverlap込みで切り出し、サブバンド相互相関で位置合わせしてOLA
    """
    if z is None: return np.array([])

    device = z.device
    w_ceil_flat = w_ceil.squeeze()
    
    full_complex_spec = None
    prev_raw_tail = None      # OLA用の前回の末尾(重複部)
    prev_chunk_complex = None # シフト探索用の前回の複素スペクトル全体(または末尾)
    
    current_ph_idx = 0
    current_z_frame = 0
    ratio = None
    
    with torch.no_grad():
        for count in chunk_counts:
            # 1. この文節の正味の長さ (zフレーム)
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # 2. 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            # 3. デコード
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    # アップサンプリング率の推定
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # --- シフト探索 (サブバンド相関) ---
                shift = 0
                if prev_chunk_complex is not None:
                    # 前回の末尾と今回の先頭を使ってシフト量を計算
                    # ※ find_best_frame_shift_subband は直前に定義した関数
                    shift = find_best_frame_shift_subband(
                        prev_chunk_complex, 
                        complex_chunk, 
                        search_range
                    )
                
                # --- OLA処理 ---
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                # シフト適用 (簡単のため、今回は複素スペクトル全体をシフトして切り出し位置を調整)
                # OLAに必要な部分を抽出
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    # 前回の末尾 (prev_raw_tail) と 今回の先頭 (curr_ref) をクロスフェード
                    
                    # shift > 0 (未来へズレる) -> 今回の開始を遅らせる (先頭を削る)
                    # shift < 0 (過去へズレる) -> 今回の開始を早める (前回の末尾に被せる深さを増やす...が、
                    # ここでは単純に「今回生成された波形」を shift 分だけずらして結合します)
                    
                    # 簡易実装: shift分だけ complex_chunk の読み出し開始位置をずらす
                    # shift正: 今回の波形が遅れているべき -> 先頭を余分に捨てる or 前回の末尾を伸ばす?
                    # ここでは「位相が合う点」を探したので、complex_chunk を shift シフトさせれば合うはず。
                    
                    # shift の符号定義: find_best... で「前に対して次がどれだけズレているか」
                    # positive shift: 次が遅れている -> 次の波形を左(過去)に寄せる必要がある?
                    # いや、相互相関のピーク位置 s は、f(t) と g(t+s) がマッチすることを意味します。
                    # なので、g (今回の波形) を -s だけずらす(スライスする)のが正解。
                    
                    # 安全マージンをとってスライス
                    start_idx = 0
                    if shift != 0:
                        # shift が正なら、今回の波形の後ろの方がマッチする -> 先頭を削る
                        if shift > 0:
                            start_idx = shift
                        # shift が負なら、今回の波形の手前がマッチする -> 本来はパディングが必要だが、
                        # 探索範囲が狭いので 0 スタートにする(無視)か、前回の末尾を削る等の調整が必要。
                        # ここでは簡易的に start_idx = 0 (負の場合は補正なし) とします
                        else:
                            start_idx = 0
                    
                    curr_ref = complex_chunk[..., start_idx : start_idx + spec_overlap_len]
                    
                    # クロスフェード長
                    cross_len = min(prev_raw_tail.shape[-1], curr_ref.shape[-1])
                    
                    if cross_len > 0:
                        print("Cross")
                        alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                        
                        # 前回の末尾 vs 今回の先頭
                        merged = prev_raw_tail[..., :cross_len] * (1 - alpha) + curr_ref[..., :cross_len] * alpha
                        
                        # 結合: [全体] + [マージ部] + [今回の残り]
                        full_complex_spec = torch.cat([
                            full_complex_spec[..., :-cross_len], # 前回のマージ部手前まで
                            merged,
                            complex_chunk[..., start_idx + cross_len:]
                        ], dim=-1)
                    else:
                        # 重なりが確保できない場合は単純結合
                        print("Pass")
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)

                # 次回用に保存
                # 次のオーバーラップ期待値
                next_expected_overlap = int(z_overlap_frames * ratio)
                
                # シフト計算用に「今回の生成結果そのもの」を保存
                prev_chunk_complex = complex_chunk
                
                # OLA用に「今回の末尾」を保存
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            # ポインタ更新
            current_ph_idx += count
            current_z_frame = z_end_nominal # 正味分だけ進める
            if current_z_frame >= z.shape[2]: break

    if full_complex_spec is None: return np.array([])
    return istft_finalize(net_g, full_complex_spec)

# ==========================================
# 4. Cond 4: 全体一括生成 (共有z版)
# ==========================================
def synthesize_cond4_shared(net_g, z, g):
    """
    共有された z をそのまま一括デコード (Topline)
    """
    if z is None: return np.array([])
    
    with torch.no_grad():
        _, _, spec, phase = net_g.dec(z, g=g)
        full_complex_spec = spec * torch.exp(1j * phase)
        
    return istft_finalize(net_g, full_complex_spec)

In [10]:
import torch
import torch.fft

def find_best_frame_shift_subband(prev_complex_spec, curr_complex_spec, search_range=2):
    """
    サブバンド（内部Zフレーム分割）単位での位相相関を用いて最適なフレームシフトを検出する
    
    Args:
        prev_complex_spec (Tensor): 前のセグメントの複素スペクトル [B, Subbands, Freq, Time]
        curr_complex_spec (Tensor): 次のセグメントの複素スペクトル [B, Subbands, Freq, Time]
        search_range (int): 探索範囲（サンプル数）
    
    Returns:
        int: 最適なシフト量 (負の値は過去方向、正の値は未来方向へのズレ)
    """
    # 1. 比較対象の抽出
    # ユーザー要件: 「細かい単位(Subbands)の一番後ろ」と「次のセグメントの一番前」
    
    # prev: 最後の時間フレームの、最後のサブバンド (Shape: [B, Freq])
    # Subbands次元(-3)の最後(-1)、Time次元(-1)の最後(-1)
    target_prev = prev_complex_spec[..., -1, :, -1] 
    
    # curr: 最初の時間フレームの、最初のサブバンド (Shape: [B, Freq])
    # Subbands次元(-3)の最初(0)、Time次元(-1)の最初(0)
    target_curr = curr_complex_spec[..., 0, :, 0]

    # 2. 周波数領域での相互相関 (Cross-Correlation via FFT)
    # 定理: Time_Corr(a, b) = iFFT( FFT(a) * conj(FFT(b)) )
    # ここでの入力は既に FFT された状態（複素スペクトル）とみなします
    
    # 片方の共役を取り、積をとる
    cross_spectrum = target_prev * torch.conj(target_curr)
    
    # iFFTして時間領域の相関関数に戻す (実数信号を仮定して irfft を使用)
    # dim=-1 (周波数方向) に対して逆変換を行います
    # 出力サイズは n_fft ( = (Freq-1)*2 ) になります
    xcorr = torch.fft.irfft(cross_spectrum, dim=-1)

    # 3. ピーク探索
    # xcorrのインデックス 0 がシフト0、インデックス 1 がシフト+1、
    # インデックス -1 (末尾) がシフト-1 に対応します。
    
    batch_size = xcorr.shape[0]
    best_shifts = []

    for b in range(batch_size):
        xc = xcorr[b]
        
        # 探索範囲のスコアを取得
        # 0周辺と、末尾(負のシフト)周辺を見ます
        
        # 候補: 0, 1, ..., range と、 -1, -2, ..., -range
        # Pythonのリストスライスで取得しやすいように配置換え（fftshift的な処理）も可能ですが、
        # ここではインデックスを直接見に行きます。
        
        best_val = -float('inf')
        best_shift = 0
        
        # 0 (No shift)
        val = xc[0].item()
        if val > best_val:
            best_val = val
            best_shift = 0
            
        for s in range(1, search_range + 1):
            # Positive shift (+s)
            val_pos = xc[s].item()
            if val_pos > best_val:
                best_val = val_pos
                best_shift = s
            
            # Negative shift (-s) -> index is N-s
            val_neg = xc[-s].item()
            if val_neg > best_val:
                best_val = val_neg
                best_shift = -s
        
        best_shifts.append(best_shift)

    # バッチ内の最頻値、または平均を採用（ここでは最頻値を採用）
    # ※ バッチサイズ1の推論時はそのまま要素0を返します
    if len(best_shifts) == 1:
        return best_shifts[0]
    
    # 多数決
    from collections import Counter
    shift_counts = Counter(best_shifts)
    most_common_shift = shift_counts.most_common(1)[0][0]
    
    return most_common_shift

In [14]:
# メインループ内での呼び出し例
# (raw_text, sid 等は設定済みとします)
phonemizer = Phonemizer()
# ==========================================
# 実行準備
# ==========================================

# 生成パラメータ
noise_scale = 1.0
noise_scale_w = 1.0
length_scale = 1.0
# (以下、メインループ内での呼び出し例)
# ...
raw_text = "ディーはエーより後ってことだよね" #(original_transcriptions_summary.txt等から取得)
#x_tst = stn_tst.to(device).unsqueeze(0)
#x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
sid = torch.LongTensor([371]).to(device)
# 1. 共通潜在表現の準備 (Cond 4相当の計算)
z, w_ceil, g, chunk_counts = prepare_shared_latents(net_g, raw_text, sid, hps, phonemizer)

if z is not None:
    # Cond 1: (これだけは独立生成が必要)
    audio_c1 = measure_and_print("Cond 1", synthesize_cond1_bunsetsu_M, net_g, raw_text, sid, hps, phonemizer)

    # Cond 2: 共有zを使用
    audio_c2 = measure_and_print("Cond 2 (Shared Z)", synthesize_cond2_shared, net_g, z, w_ceil, g, chunk_counts)

    # Cond 3: 共有zを使用
    audio_c3 = measure_and_print("Cond 3 (Shared Z)", synthesize_cond3_shared, net_g, z, w_ceil, g, chunk_counts, 1, 1)

    # Cond 4: 共有zを使用
    audio_c4 = measure_and_print("Cond 4 (Shared Z)", synthesize_cond4_shared, net_g, z, g)




print("条件１")
display(Audio(audio_c1, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c1 min: {np.min(audio_c1):.4f}, max: {np.max(audio_c1):.4f}")
print("条件２")
display(Audio(audio_c2, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c2 min: {np.min(audio_c2):.4f}, max: {np.max(audio_c2):.4f}")
print("条件３")
display(Audio(audio_c3, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c3 min: {np.min(audio_c3):.4f}, max: {np.max(audio_c3):.4f}")

print("条件4")
display(Audio(audio_c4, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c4 min: {np.min(audio_c4):.4f}, max: {np.max(audio_c4):.4f}")

# 生成パラメータ
noise_scale = 0.1
noise_scale_w = 0.667
length_scale = 1.0
raw_text = "ディーはエーより後ってことだよね" #(original_transcriptions_summary.txt等から取得)
#x_tst = stn_tst.to(device).unsqueeze(0)
#x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
sid = torch.LongTensor([371]).to(device)
# 1. 共通潜在表現の準備 (Cond 4相当の計算)
z, w_ceil, g, chunk_counts = prepare_shared_latents(net_g, raw_text, sid, hps, phonemizer)

if z is not None:
    # Cond 1: (これだけは独立生成が必要)
    audio_c1 = measure_and_print("Cond 1", synthesize_cond1_bunsetsu_M, net_g, raw_text, sid, hps, phonemizer)

    # Cond 2: 共有zを使用
    audio_c2 = measure_and_print("Cond 2 (Shared Z)", synthesize_cond2_shared, net_g, z, w_ceil, g, chunk_counts)

    # Cond 3: 共有zを使用
    audio_c3 = measure_and_print("Cond 3 (Shared Z)", synthesize_cond3_shared, net_g, z, w_ceil, g, chunk_counts, 1, 1)

    # Cond 4: 共有zを使用
    audio_c4 = measure_and_print("Cond 4 (Shared Z)", synthesize_cond4_shared, net_g, z, g)




print("条件１")
display(Audio(audio_c1, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c1 min: {np.min(audio_c1):.4f}, max: {np.max(audio_c1):.4f}")
print("条件２")
display(Audio(audio_c2, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c2 min: {np.min(audio_c2):.4f}, max: {np.max(audio_c2):.4f}")
print("条件３")
display(Audio(audio_c3, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c3 min: {np.min(audio_c3):.4f}, max: {np.max(audio_c3):.4f}")

print("条件4")
display(Audio(audio_c4, rate=hps.data.sampling_rate, normalize=False))
print(f"audio_c4 min: {np.min(audio_c4):.4f}, max: {np.max(audio_c4):.4f}")

NameError: name 'measure_and_print' is not defined

In [32]:
import os

# ==========================================
# 1. 日本語テキスト (書き起こし) の定義
# ==========================================
raw_input_text = """
uudb/tts1/data/test/wav/FJK_C051_118.wav
  Original: でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた

uudb/tts1/data/test/wav/FJK_C051_170.wav
  Original: やっぱ、やっぱディーが最後かな

uudb/tts1/data/test/wav/FKC_C031_002.wav
  Original: [うんとね]多分あたし一番持ってる気がするって感じ

uudb/tts1/data/test/wav/FMS_C051_072.wav
  Original: そうだよね、うんうん

uudb/tts1/data/test/wav/FMT_C041_134.wav
  Original: 何だそりゃ

uudb/tts1/data/test/wav/FMT_C041_259.wav
  Original: ディーはエーより後ってことだよね

uudb/tts1/data/test/wav/FTH_C004_044.wav
  Original: [えっとね]じゃあシー、行くね

uudb/tts1/data/test/wav/FTH_C005_152.wav
  Original: [あ]じゃあ、最初は、エーの[その]、交番の外にサラリーマンがいて

uudb/tts1/data/test/wav/FTS_C002_107.wav
  Original: [あ]ちゃんと入ってないんだ、[あー]

uudb/tts1/data/test/wav/FTS_C002_175.wav
  Original: [あー]、オッケーオッケー

uudb/tts1/data/test/wav/FTS_C004_126.wav
  Original: そうね、えっみたいな感じだよね

uudb/tts1/data/test/wav/FTS_C006_050.wav
  Original: なるほど、わかりました

uudb/tts1/data/test/wav/FTS_C007_137.wav
  Original: [あー]、見てるんだ

uudb/tts1/data/test/wav/FUE_C033_134.wav
  Original: [え]、わかんなくなってきた

uudb/tts1/data/test/wav/FYH_C042_090.wav
  Original: この、お母さんがすごい、あんまり上品でない食べ方をしてるのね

uudb/tts1/data/test/wav/FYH_C043_064.wav
  Original: はい、いいですか
"""

# ==========================================
# 2. 話者ID (SID) の参照データ定義
# ==========================================
sid_reference_data = """
uudb/tts1/data/test/wav/FJK_C051_118.wav|368|d e m o [ n a N k a k o: ] s o: s u r u t o sp m a t a m a t a a Q t a k a k u n a Q t a Q t e k o n o n a g a s i m a k a N t o k u g a y u Q t e r u N d a y o n e m a t a
uudb/tts1/data/test/wav/FJK_C051_170.wav|368|y a Q p a sp y a Q p a d i: g a s a i g o k a n a
uudb/tts1/data/test/wav/FKC_C031_002.wav|369|[ u N t o n e ] t a b u N a t a s i i t i b a N m o Q t e r u k i g a s u r u Q t e k a N z i
uudb/tts1/data/test/wav/FMS_C051_072.wav|370|s o: d a y o n e sp u N u N
uudb/tts1/data/test/wav/FMT_C041_134.wav|371|n a N d a s o ry a
uudb/tts1/data/test/wav/FMT_C041_259.wav|371|d i: w a e: y o r i a t o Q t e k o t o d a y o n e
uudb/tts1/data/test/wav/FTH_C004_044.wav|375|[ e Q t o n e ] zy a: s i: sp i k u n e
uudb/tts1/data/test/wav/FTH_C005_152.wav|375|[ a ] zy a: sp s a i sy o w a sp e: n o [ s o n o ] sp k o: b a N n o s o t o n i s a r a r i: m a N g a i t e
uudb/tts1/data/test/wav/FTS_C002_107.wav|376|[ a ] ch a N t o h a i Q t e n a i N d a sp [ a: ]
uudb/tts1/data/test/wav/FTS_C002_175.wav|376|[ a: ] sp o Q k e: o Q k e:
uudb/tts1/data/test/wav/FTS_C004_126.wav|376|s o: n e sp e Q m i t a i n a k a N z i d a y o n e
uudb/tts1/data/test/wav/FTS_C006_050.wav|376|n a r u h o d o sp w a k a r i m a s i t a
uudb/tts1/data/test/wav/FTS_C007_137.wav|376|[ a: ] sp m i t e r u N d a
uudb/tts1/data/test/wav/FUE_C033_134.wav|378|[ e ] sp w a k a N n a k u n a Q t e k i t a
uudb/tts1/data/test/wav/FYH_C042_090.wav|379|k o n o sp o k a: s a N g a s u g o i sp a N m a r i zy o: h i N d e n a i t a b e k a t a o s i t e r u n o n e
uudb/tts1/data/test/wav/FYH_C043_064.wav|379|h a i sp i: d e s u k a
"""

# ==========================================
# 3. データの紐付けとリスト生成処理
# ==========================================

def prepare_execution_list(raw_text, sid_data):
    # 1. SIDマップの作成 (ファイル名 -> SID)
    filename_to_sid = {}
    for line in sid_data.strip().split('\n'):
        line = line.strip()
        if not line: continue
        parts = line.split('|')
        if len(parts) >= 2:
            path = parts[0]
            sid = parts[1]
            fname = os.path.basename(path)
            filename_to_sid[fname] = int(sid)

    print(f"Loaded SID mapping for {len(filename_to_sid)} files.")

    # 2. テキストデータのパースと結合
    execution_lines = []
    current_wav = None
    
    for line in raw_text.strip().split('\n'):
        line = line.strip()
        if not line: continue
        
        if line.endswith('.wav'):
            current_wav = line
        elif line.startswith('Original:'):
            if current_wav:
                text = line.replace('Original:', '').strip()
                fname = os.path.basename(current_wav)
                
                # SIDの引き当て
                if fname in filename_to_sid:
                    sid_val = filename_to_sid[fname]
                    # フォーマット: パス|SID|テキスト
                    formatted_line = f"{current_wav}|{sid_val}|{text}"
                    execution_lines.append(formatted_line)
                else:
                    print(f"Warning: No SID found for {fname}, skipping.")
                
                current_wav = None

    return execution_lines

# リストの生成
lines = prepare_execution_list(raw_input_text, sid_reference_data)

print(f"Prepared {len(lines)} tasks.")
# 確認用出力 (最初の1件)
if lines:
    print("Sample task:", lines[0])

# ==========================================
# 4. メイン実行ループ
# ==========================================
# (以前のコードのループ部分をそのまま使用します)

print("Starting measurement loop...")

# チャンクサイズ設定 (Cond 2, 3用)
mora_chunk = 1

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    # 読み込み
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\nProcessing: {filename} (SID: {spk_id_val})")
    print(f"Text: {raw_text_jp}")

    try:
        # Cond 1: モーラ単位単純接続
        audio_c1 = measure_and_print(
            "Cond 1 (Naive Mora Concat)", 
            synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)

        # Cond 2: スペクトログラム単純接続
        audio_c2 = measure_and_print(
            f"Cond 2 (Spec Naive Concat, chunk={mora_chunk})", 
            synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio_c2)

        # Cond 3: 相互相関OLA
        audio_c3 = measure_and_print(
            f"Cond 3 (Correlation OLA, chunk={mora_chunk})", 
            synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 
            1, 2   # overlap=5, search=2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio_c3)

        # Cond 4: 全体一括 (Baseline)
        audio_c4 = measure_and_print(
            "Cond 4 (Full Context Baseline)", 
            synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio_c4)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

print("All tasks finished.")

Loaded SID mapping for 16 files.
Prepared 16 tasks.
Sample task: uudb/tts1/data/test/wav/FJK_C051_118.wav|368|でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
Starting measurement loop...

Processing: FJK_C051_118 (SID: 368)
Text: でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
--- Cond 1 (Naive Mora Concat) ---
Audio duration: 7.58 seconds
Elapsed time: 0.3326 seconds
Real Time Factor (RTF): 0.0439
------------------------------
--- Cond 2 (Spec Naive Concat, chunk=1) ---
Audio duration: 6.27 seconds
Elapsed time: 0.2241 seconds
Real Time Factor (RTF): 0.0358
------------------------------
--- Cond 3 (Correlation OLA, chunk=1) ---
Audio duration: 6.28 seconds
Elapsed time: 0.1921 seconds
Real Time Factor (RTF): 0.0306
------------------------------
--- Cond 4 (Full Context Baseline) ---
Audio duration: 5.95 seconds
Elapsed time: 0.0321 seconds
Real Time Factor (RTF): 0.0054
------------------------------

Processing: FJK_C051_170 (SID: 368)
Text: やっぱ、やっぱディーが最後かな
--- Cond 1 (Naive Mora 

In [15]:
get_bunsetsu_chunks_mecab('あらゆる現実を、全て自分の方へ捻じ曲げたのだ。' , phonemizer)

['a r a y u r u',
 'g e N z i t u o sp',
 's u b e t e',
 'z i b u N n o',
 'k a t a e',
 'n e z i m a g e t a n o d a sp']